In [1]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import folium
from shapely.geometry import box

In [2]:
lat = float(input("Enter Latitude:  "))
lon = float(input("Enter Longitude: "))
 
print("\n⏳ Loading road network...")

Enter Latitude:   19.100485560262815
Enter Longitude:  72.82034152333493



⏳ Loading road network...



⏳ Loading road network...


In [3]:
cf = '["highway"~"motorway|trunk|primary|secondary|tertiary|residential"]'
G = ox.graph_from_point(
    (lat, lon),
    dist=5000,
    custom_filter=cf
)

In [ ]:
BUFFER = 0.05   # ~5 km in degrees
 
print("⏳ Loading road traffic data (bbox-filtered)...")
roads = gpd.read_file(
    r"C:\Users\krish\Downloads\ROAD 1\ROAD.geojson",
    bbox=(lon - BUFFER, lat - BUFFER, lon + BUFFER, lat + BUFFER)
)
roads = roads.to_crs(epsg=4326)
print(f"   ✅ Loaded {len(roads)} road segments")
 
# ── Show available columns so you can verify the density column name ──
print("   Road columns:", roads.columns.tolist())

In [ ]:
# ⚠️  CHANGE THIS to match your actual density/traffic column name
DENSITY_COL = "density"   # e.g. "traffic_volume", "congestion", "flow"
 
if DENSITY_COL not in roads.columns:
    print(f"\n⚠️  Column '{DENSITY_COL}' not found. "
          f"Available: {roads.columns.tolist()}"
          "\n   Falling back to road-type-only weights.\n")
    roads[DENSITY_COL] = 1.0   # neutral fallback

In [ ]:
print("⏳ Joining traffic density to graph edges...")
 
edges_gdf = ox.graph_to_gdfs(G, nodes=False).reset_index()
edges_gdf = edges_gdf.to_crs(epsg=4326)
 
edges_joined = gpd.sjoin_nearest(
    edges_gdf[["u", "v", "key", "geometry"]],
    roads[["geometry", DENSITY_COL]],
    how="left",
    max_distance=0.001   # ~100 m tolerance
)
 
# Build lookup dict  (u, v, key) → density value
density_lookup = {
    (row["u"], row["v"], row["key"]): row[DENSITY_COL]
    for _, row in edges_joined.iterrows()
}

In [ ]:
valid_densities = roads[DENSITY_COL].dropna()
max_density = valid_densities.quantile(0.95) if len(valid_densities) > 0 else 1.0
if max_density == 0:
    max_density = 1.0

In [ ]:
BASE_FACTOR = {
    "motorway":    1.0,
    "trunk":       1.0,
    "primary":     1.2,
    "secondary":   1.4,
    "tertiary":    1.6,
    "residential": 3.5,   # strongly penalise back-lanes
}
 
for u, v, k, data in G.edges(keys=True, data=True):
    length  = data.get("length", 1)
    highway = data.get("highway", "unclassified")
 
    if isinstance(highway, list):
        highway = highway[0]
 
    base = BASE_FACTOR.get(highway, 1.5)
 
    # Real density from GeoJSON (scaled 1.0 → 3.0)
    raw = density_lookup.get((u, v, k), None)
    if raw is not None and not (raw != raw):   # not NaN
        density_factor = 1.0 + 2.0 * (float(raw) / max_density)
        density_factor = min(density_factor, 4.0)   # cap at 4×
    else:
        density_factor = 1.0
 
    data["travel_time"] = length * base * density_factor
 

In [ ]:
print("⏳ Loading hospitals...")
hospitals = gpd.read_file(r"C:\Users\krish\Downloads\Hospital 1.geojson")
hospitals = hospitals.to_crs(epsg=4326)
 
hospitals["node"] = hospitals.apply(
    lambda row: ox.nearest_nodes(G, row.geometry.x, row.geometry.y),
    axis=1
)
 

In [ ]:
orig_node = ox.nearest_nodes(G, lon, lat)
 
best_time     = float("inf")
best_hospital = None
best_route    = None
 
print("⏳ Computing shortest paths to all hospitals...")
for idx, row in hospitals.iterrows():
    try:
        t = nx.shortest_path_length(G, orig_node, row["node"], weight="travel_time")
        if t < best_time:
            best_time     = t
            best_hospital = row
            best_route    = nx.shortest_path(G, orig_node, row["node"], weight="travel_time")
    except nx.NetworkXNoPath:
        continue
    except Exception:
        continue
 
if best_hospital is None:
    print("❌ No reachable hospital found. Try increasing dist= in graph_from_point.")
else:
    time_min    = best_time / 60
    distance_km = nx.shortest_path_length(
        G, orig_node, best_hospital["node"], weight="length"
    ) / 1000
 
    print("\n🚑 FASTEST HOSPITAL (TRAFFIC-AWARE)")
    print("─" * 40)
    print(f"  Name         : {best_hospital.get('name', 'Unknown')}")
    print(f"  Distance     : {round(distance_km, 2)} km")
    print(f"  Est. Time    : {round(time_min, 2)} min  (traffic-adjusted)")
    print("─" * 40)

In [ ]:
    # ─────────────────────────────────────────
    # 8. BUILD FOLIUM MAP
    # ─────────────────────────────────────────
    m = folium.Map(location=[lat, lon], zoom_start=15, tiles="CartoDB positron")
 
    # User location
    folium.Marker(
        [lat, lon],
        popup="📍 Your Location",
        tooltip="You are here",
        icon=folium.Icon(color="blue", icon="user", prefix="fa")
    ).add_to(m)
 
    # Best hospital
    folium.Marker(
        [best_hospital.geometry.y, best_hospital.geometry.x],
        popup=f"🏥 {best_hospital.get('name', 'Hospital')}<br>"
              f"{round(distance_km, 2)} km | ~{round(time_min, 1)} min",
        tooltip=best_hospital.get("name", "Hospital"),
        icon=folium.Icon(color="red", icon="plus", prefix="fa")
    ).add_to(m)
 
    # Route polyline
    route_coords = [(G.nodes[n]["y"], G.nodes[n]["x"]) for n in best_route]
    folium.PolyLine(
        route_coords,
        color="#00cc66",
        weight=6,
        opacity=0.85,
        tooltip=f"~{round(time_min, 1)} min  |  {round(distance_km, 2)} km"
    ).add_to(m)
 
    # All other hospitals (grey markers)
    for _, row in hospitals.iterrows():
        if row.get("name") != best_hospital.get("name"):
            folium.CircleMarker(
                [row.geometry.y, row.geometry.x],
                radius=6,
                color="#888",
                fill=True,
                fill_opacity=0.5,
                tooltip=row.get("name", "Hospital")
            ).add_to(m)
 
    # Save map
    OUTPUT_PATH = r"C:\Users\PARTH\Documents\hospital_route.html"
    m.save(OUTPUT_PATH)
    print(f"\n🗺️  Map saved → {OUTPUT_PATH}")
    m   # display inline in Jupyter
 